## Подготовка данных

Используем датасет **Iris** — задачу кластеризации с тремя классами цветов ириса.

In [1]:
from sklearn.datasets import load_iris
import numpy as np

iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
class_names = iris.target_names

print(f"\nРаспределение классов:")
for i, name in enumerate(class_names):
    count = int(np.sum(y == i))
    print(f"  {i} ({name}): {count} ({100*count/len(y):.1f}%)")


Распределение классов:
  0 (setosa): 50 (33.3%)
  1 (versicolor): 50 (33.3%)
  2 (virginica): 50 (33.3%)


# Нормализация признаков

In [2]:
mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / std

## Обучение GMM

In [3]:
import sys
sys.path.insert(0, '.')
from gmm import GMM

np.random.seed(42)
gmm = GMM(k=3, max_iter=100, tol=1e-4)
gmm.fit(X_scaled)



In [8]:
np.random.seed(42)
gmm_trace = GMM(k=3, max_iter=100, tol=1e-4)
gmm_trace.weights = np.ones(gmm_trace.k) / gmm_trace.k
gmm_trace.means = X_scaled[np.random.choice(X_scaled.shape[0], gmm_trace.k, replace=False)]
gmm_trace.covs = np.array([np.cov(X_scaled, rowvar=False) for _ in range(gmm_trace.k)])

log_likelihood_old = None
print(f"{'Итерация':<12} {'Log-Likelihood':<20} {'Delta':<15}")
for iteration in range(gmm_trace.max_iter):
    responsibilities = gmm_trace._e_step(X_scaled)
    gmm_trace._m_step(X_scaled, responsibilities)

    ll = gmm_trace._log_likelihood(X_scaled)
    delta = abs(ll - log_likelihood_old) if log_likelihood_old is not None else float('inf')
    print(f"{iteration:<12} {ll:<20.6f} {delta:<15.2e}")

    if log_likelihood_old is not None and delta < gmm_trace.tol:
        print(f"\nСошлось за {iteration + 1} итераций EM")
        break
    log_likelihood_old = ll

Итерация     Log-Likelihood       Delta          
0            -453.643401          inf            
1            -433.103456          2.05e+01       
2            -405.002204          2.81e+01       
3            -368.719938          3.63e+01       
4            -320.821503          4.79e+01       
5            -317.084639          3.74e+00       
6            -316.778923          3.06e-01       
7            -316.513089          2.66e-01       
8            -316.175506          3.38e-01       
9            -315.638661          5.37e-01       
10           -314.732366          9.06e-01       
11           -313.287648          1.44e+00       
12           -311.047329          2.24e+00       
13           -307.588329          3.46e+00       
14           -303.204949          4.38e+00       
15           -298.790873          4.41e+00       
16           -295.189630          3.60e+00       
17           -292.809676          2.38e+00       
18           -291.518041          1.29e+00       


## Оценка качества

In [4]:
from sklearn.metrics import adjusted_rand_score, silhouette_score

def compute_log_likelihood(gmm, X):
    log_likelihood = 0
    for k in range(gmm.k):
        log_likelihood += gmm.weights[k] * gmm._gaussian_pdf(X, gmm.means[k], gmm.covs[k])
    return np.sum(np.log(log_likelihood))

responsibilities = gmm._e_step(X_scaled)
labels_pred = np.argmax(responsibilities, axis=1)

ari = adjusted_rand_score(y, labels_pred)
sil = silhouette_score(X_scaled, labels_pred)

print(f"Adjusted Rand Index (ARI): {ari:.4f}")
print(f"Silhouette Score:          {sil:.4f}")

Adjusted Rand Index (ARI): 0.9039
Silhouette Score:          0.3742


## Сравнение со sklearn

Сравним с `GaussianMixture` из sklearn (k=3, те же параметры инициализации).

In [10]:
from sklearn.mixture import GaussianMixture

sklearn_gmm = GaussianMixture(n_components=3, max_iter=100, tol=1e-4, random_state=42, init_params='random')
sklearn_gmm.fit(X_scaled)
labels_sklearn = sklearn_gmm.predict(X_scaled)

ari_sklearn = adjusted_rand_score(y, labels_sklearn)
sil_sklearn = silhouette_score(X_scaled, labels_sklearn)
ll_sklearn = sklearn_gmm.score(X_scaled) * X_scaled.shape[0]
print(sklearn_gmm.n_iter_)

np.random.seed(42)
gmm_our = GMM(k=3, max_iter=100, tol=1e-4)
gmm_our.fit(X_scaled)
resp_our = gmm_our._e_step(X_scaled)
labels_our = np.argmax(resp_our, axis=1)
ari_our = adjusted_rand_score(y, labels_our)
sil_our = silhouette_score(X_scaled, labels_our)
ll_our = compute_log_likelihood(gmm_our, X_scaled)

print(f"{'Метрика':<25} {'самописный GMM':<20} {'sklearn GMM':<20}")
print(f"{'Log-Likelihood':<25} {ll_our:<20.2f} {ll_sklearn:<20.2f}")
print(f"{'ARI':<25} {ari_our:<20.4f} {ari_sklearn:<20.4f}")
print(f"{'Silhouette Score':<25} {sil_our:<20.4f} {sil_sklearn:<20.4f}")

21
Метрика                   самописный GMM       sklearn GMM         
Log-Likelihood            -290.53              -299.89             
ARI                       0.9039               0.5608              
Silhouette Score          0.3742               0.3139              


In [ ]:
print("Распределение объектов: мой GMM vs sklearn GMM\n")
print(f"{'Класс':<15} {'самописный GMM':>30}   {'sklearn GMM':>30}")
for i, name in enumerate(class_names):
    our_dist = ", ".join(
        f"k{j}:{int(np.sum((y==i)&(labels_our==j)))}" for j in range(3)
    )
    sk_dist = ", ".join(
        f"k{j}:{int(np.sum((y==i)&(labels_sklearn==j)))}" for j in range(3)
    )
    print(f"{name:<15} {our_dist:>30}   {sk_dist:>30}")

Распределение объектов: наш GMM vs sklearn GMM

Класс                           самописный GMM                      sklearn GMM
setosa                       k0:0, k1:50, k2:0                k0:0, k1:45, k2:5
versicolor                   k0:45, k1:0, k2:5                k0:48, k1:0, k2:2
virginica                    k0:0, k1:0, k2:50                k0:50, k1:0, k2:0
